# HW3 — Two retrievers and a model: what RAG actually sends

**Name:Madiyar Tokbergen**
**Student ID: S23068824**
**Group: 10**
**Repository: https://github.com/AkamenYY/css4007-hw3**

## AI tool disclosure

State which AI tools you used and for what. Expected and fine; undisclosed use is not. I used Claude to understand the assignment, set up the environment and get starter code. All results, hand calculations and analysis are based on my own runs.

>

Run this notebook **top to bottom** before you push. A cell with no saved output earns nothing.
The instructions for every section are in `README.md`.

In [1]:
import os, json, re
import numpy as np
from dotenv import load_dotenv
load_dotenv()

print("OpenAI key:", bool(os.getenv("OPENAI_API_KEY")))
print("HF token:", bool(os.getenv("HF_TOKEN")))

corpus  = [json.loads(l) for l in open("data/corpus.jsonl", encoding="utf-8")]
queries = json.load(open("data/queries.json", encoding="utf-8"))["queries"]
qa      = json.load(open("data/qa.json", encoding="utf-8"))["questions"]

ids   = [c["id"] for c in corpus]
texts = [c["text"] for c in corpus]
by_id = {c["id"]: c for c in corpus}
print(len(corpus), "chunks |", len(queries), "queries |", len(qa), "questions")

OpenAI key: True
HF token: True
74 chunks | 8 queries | 6 questions


In [2]:
for c in corpus:
    if c["id"].startswith("GO"):
        print(f"--- {c['id']} [{c['lang']}] {c['title']}\n{c['text']}\n")

--- GO-01 [en] About the need-based study grant 2026
The need-based study grant 2026 is run by the Grant Office for full-time undergraduate students. It exists so that a student from a low-income household can pay for study costs without taking on paid work that harms their studies. One grant is awarded per student per academic year. The grant is a gift, not a loan: it does not have to be paid back unless the student withdraws early, as explained in the section on academic progress.

--- GO-02 [en] Who qualifies
An applicant qualifies for the 2026 grant when all three conditions hold at the time of the decision: their cumulative GPA is at least 2.67 on the 4.0 scale; their household is in income band 1 or income band 2; and both a transcript and a copy of the national id card are on file with the Grant Office. Meeting two conditions out of three is not enough. An applicant who is not on the Grant Office record does not qualify, whatever their message says.

--- GO-03 [en] Income bands


**Notes on the handbook (GO chunks):**
- **Archived rule:** GO-05 is the old 2025 rule (min GPA 3.0, one payment in November). The current rule is GO-02: min GPA 2.67. GO-05 uses the same words as the question "minimum GPA for the grant".
- **Similar forms:** GR-104 = application, GR-210 = income declaration, GR-305 = appeal.
- **Non-English chunks:** GO-14 [kk] = application deadline (1 Sep – 15 Oct); GO-15 [ru] = grant is paid only to the student's own account, not to parents; GO-23 [ru] = appeal summary.
- **Master's students:** GO-20 says they have a separate scheme that the handbook does not describe, so the correct answer for them is "I don't know".

---
# Part A — relevance by hand (1 pt)

| | Text |
|---|---|
| D1 | Large language models can generate text and answer questions. |
| D2 | Retrieval-Augmented Generation retrieves relevant documents before generating an answer. |
| D3 | Embeddings represent text as numerical vectors. |
| Query | How does RAG retrieve information? |

`k1 = 1.5`, `b = 0.75`, `idf(t) = ln((N − n(t) + 0.5) / (n(t) + 0.5))`. Tokenise as README §3 says.



## A1 (a) — BM25 as written

**Tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, generate, text, and, answer, questions | 9 |
| D2 | retrieval, augmented, generation, retrieves, relevant, documents, before, generating, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**avgdl** = (9 + 10 + 6) / 3 = 25 / 3 ≈ 8.333

**Query terms:** how, does, rag, retrieve, information
**TF, DF and IDF of each query term:**

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---|---|---|---|---|
| how | 0 | 0 | 0 | 0 | ln(3.5/0.5) = ln(7) ≈ 1.946 |
| does | 0 | 0 | 0 | 0 | ln(7) ≈ 1.946 |
| rag | 0 | 0 | 0 | 0 | ln(7) ≈ 1.946 |
| retrieve | 0 | 0 | 0 | 0 | ln(7) ≈ 1.946 |
| information | 0 | 0 | 0 | 0 | ln(7) ≈ 1.946 |

**BM25 of each document** (write out the sum for every document, one line per term that contributes):

>

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | 0 + 0 + 0 + 0 + 0 = 0| tie (1–3)|
| D2 | 0 + 0 + 0 + 0 + 0 = 0| tie (1–3)|
| D3 | 0 + 0 + 0 + 0 + 0 = 0| tie (1–3)|

## A1 (b) — BM25 with the stems from README §3

**Stemmed tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, gener, text, and, answer, questions | 9 |
| D2 | retriev, augmented, gener, retriev, relevant, documents, before, gener, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**avgdl** = (9 + 10 + 6) / 3 ≈ 8.333 (lengths do not change after stemming)

**Stemmed query terms:** how, doe, rag, retriev, inform

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---|---|---|---|---|
| how | 0 | 0 | 0 | 0 | ln(7) ≈ 1.946 |
| doe | 0 | 0 | 0 | 0 | ln(7) ≈ 1.946 |
| rag | 0 | 0 | 0 | 0 | ln(7) ≈ 1.946 |
| retriev | 0 | 2 | 0 | 1 | ln((3−1+0.5)/(1+0.5)) = ln(2.5/1.5) ≈ 0.511 |
| inform | 0 | 0 | 0 | 0 | ln(7) ≈ 1.946 |


**BM25 of each document**, worked:

>

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | 0 (no query term matches) | 2 (tie with D3) |
| D2 | 0.511 × 2 × 2.5 / (2 + 1.5 × (0.25 + 0.75 × 10/8.333)) = 2.555 / 3.725 ≈ 0.686 | 1 |
| D3 | 0 (no query term matches) | 2 (tie with D1) |

**Which document does BM25 retrieve first, in (a) and in (b)? Why?**
**(a)** No document is retrieved first: all three scores are 0, so it is a three-way tie. This is not an arithmetic mistake. BM25 only counts exact token matches, and none of the query tokens (how, does, rag, retrieve, information) appears in any document. D2 is clearly the relevant document, but it contains "retrieves" and "retrieval", not "retrieve", and it spells RAG out as "Retrieval-Augmented Generation" instead of using the abbreviation.                      **(b)** D2 is retrieved first (score ≈ 0.686), and D1 and D3 tie at 0. Stemming maps "retrieves", "retrieval" and "retrieve" to the same stem "retriev", so the query now matches D2 twice. "rag" still does not match, because stemming cannot connect an abbreviation to its full form. BM25 matches strings, not meanings.

>


‖Query‖ = √(0.8² + 0.2² + 0.5²) = √(0.64 + 0.04 + 0.25) = √0.93 ≈ 0.9644

| Pair | Dot product (worked) | ‖Query‖ | ‖D‖ | Cosine |
|---|---|---:|---:|---:|
| Query · D1 | 0.8·0.7 + 0.2·0.1 + 0.5·0.4 = 0.56 + 0.02 + 0.20 = 0.78 | 0.9644 | √(0.49+0.01+0.16) = √0.66 ≈ 0.8124 | 0.78 / (0.9644 × 0.8124) ≈ 0.9956 |
| Query · D2 | 0.8·0.9 + 0.2·0.3 + 0.5·0.6 = 0.72 + 0.06 + 0.30 = 1.08 | 0.9644 | √(0.81+0.09+0.36) = √1.26 ≈ 1.1225 | 1.08 / (0.9644 × 1.1225) ≈ 0.9977 |
| Query · D3 | 0.8·0.1 + 0.2·0.8 + 0.5·0.2 = 0.08 + 0.16 + 0.10 = 0.34 | 0.9644 | √(0.01+0.64+0.04) = √0.69 ≈ 0.8307 | 0.34 / (0.9644 × 0.8307) ≈ 0.4244 |

| Document | Cosine Similarity | Rank |
|---|---:|---:|
| D1 | 0.9956 | 2 |
| D2 | 0.9977 | 1 |
| D3 | 0.4244 | 3 |

**Which document is most similar to the query?**

> D2, with cosine ≈ 0.998. D1 is very close behind (≈ 0.996), and D3 is far away (≈ 0.42).

**Is the ranking the same as BM25's?** Compare it with both A1 rankings.

> Not with A1(a): there BM25 gave a three-way tie at 0, while cosine gives a clear order D2 > D1 > D3. With A1(b) the winner is the same (D2), but the rest differs: BM25 gives D1 and D3 exactly 0 and ties them, while cosine puts D1 almost level with D2 and D3 clearly last.

**Why can BM25 and embedding retrieval disagree?** Point at the words of this query and these documents.

> BM25 scores only exact token overlap. The query says "RAG", "retrieve" and "information", while D2 says "Retrieval-Augmented Generation", "retrieves" and "documents", so without stemming nothing matches, and even with stemming "rag" never matches its full form. An embedding model compares meaning, so "RAG" can sit close to "Retrieval-Augmented Generation". D1 ("language models ... answer questions") also scores high because it is on the same topic, even though it shares no word with the query. D3 is about vectors, a different topic, so it scores low.

Optional: check your Part A arithmetic here. The working above must still be written out by hand.

In [3]:
import math
Q = [0.8, 0.2, 0.5]
D = {"D1": [0.7, 0.1, 0.4], "D2": [0.9, 0.3, 0.6], "D3": [0.1, 0.8, 0.2]}
nq = math.sqrt(sum(x*x for x in Q))
for name, v in D.items():
    dot = sum(a*b for a, b in zip(Q, v))
    nd = math.sqrt(sum(x*x for x in v))
    print(f"{name}: dot={dot:.2f}  |Q|={nq:.4f}  |D|={nd:.4f}  cos={dot/(nq*nd):.4f}")

D1: dot=0.78  |Q|=0.9644  |D|=0.8124  cos=0.9956
D2: dot=1.08  |Q|=0.9644  |D|=1.1225  cos=0.9977
D3: dot=0.34  |Q|=0.9644  |D|=0.8307  cos=0.4244


---
# Part B — the same two retrievers, from libraries (1.5 pt)

Load `data/corpus.jsonl` and `data/queries.json`.

In [4]:
print(len(corpus), "chunks loaded")
for q in queries:
    print(q["id"], "|", q["text"], "| relevant:", q["relevant"])

74 chunks loaded
Q-01 | What is form GR-305 used for? | relevant: ['GO-08', 'GO-12', 'GO-23']
Q-02 | My family is poor. Is there financial help for students like me? | relevant: ['GO-01', 'GO-02', 'GO-03']
Q-03 | Өтінішті қашанға дейін тапсыру керек? | relevant: ['GO-09', 'GO-14']
Q-04 | What is the minimum GPA for the grant? | relevant: ['GO-02']
Q-05 | When was Astana renamed Nur-Sultan, and why? | relevant: ['KZ-05']
Q-06 | Where did the Soviet Union test its nuclear weapons? | relevant: ['KZ-37', 'KZ-38']
Q-07 | Can the grant be sent to my mother's card? | relevant: ['GO-15']
Q-08 | If my grades drop in the autumn, do I still get paid in February? | relevant: ['GO-17']


## B1 — BM25 with `rank_bm25`

**My tokenisation: lowercase; regex `\w+` (Unicode-aware, so Kazakh and Russian words stay whole); no stemming; no stop-word removal. A form code like "GR-305" becomes two tokens: "gr" and "305".** (lowercase? regex? stemming? stop words?)

>

In [5]:
from rank_bm25 import BM25Okapi

def tokenize(s):
    return re.findall(r"\w+", s.lower())

bm25 = BM25Okapi([tokenize(t) for t in texts], k1=1.5, b=0.75)

def bm25_top(q, k=3):
    scores = bm25.get_scores(tokenize(q))
    idx = np.argsort(scores)[::-1][:k]
    return [(ids[i], float(scores[i])) for i in idx]

def first_line(cid):
    return by_id[cid]["text"].split("\n")[0][:70]

print(bm25_top("What is form GR-305 used for?"))

[('GO-08', 15.825491699499644), ('GO-12', 8.195256931904641), ('GO-06', 8.07003113015204)]


## B2 — EmbeddingGemma (`google/embeddinggemma-300m`) with `encode_document` / `encode_query`

In [7]:
from sentence_transformers import SentenceTransformer

emb_model = SentenceTransformer("google/embeddinggemma-300m", token=os.getenv("HF_TOKEN"))
doc_emb = emb_model.encode_document(texts)

def emb_top(q, k=3):
    q_emb = emb_model.encode_query(q)
    sims = emb_model.similarity(q_emb, doc_emb)[0].numpy()
    idx = np.argsort(sims)[::-1][:k]
    return [(ids[i], float(sims[i])) for i in idx]

print(emb_top("What is form GR-305 used for?"))

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.21GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

...

2_Dense/model.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

3_Dense/model.safetensors: reconstructing file:   0%|          |  0.00B / 9.44MB            

3_Dense/model.safetensors: downloading bytes:           |  0.00B            

[('GO-08', 0.6955164670944214), ('GO-12', 0.46087244153022766), ('GO-23', 0.4504862129688263)]


## My own queries

Write them, and their `relevant` chunk ids, **before** you run them. Say which one you expect BM25 to win and which you expect embeddings to win, and why.

| Id | Query | `relevant` | Expected winner, and why |
|---|---|---|---|
| Q-09 | Who has to sign form GR-210? | GO-07 | **BM25**: the query contains the exact form code "GR-210" (tokens "gr", "210") and the word "sign", which appear in GO-07; embeddings may blur it with the other GR-forms. |
| Q-10 | Когда закрывается приём заявок на грант? | GO-09, GO-14 | **Embeddings**: the query is in Russian, but the answer is in English (GO-09) and Kazakh (GO-14), so BM25 shares no tokens with them; a multilingual embedding model can match the meaning across languages. |

In [8]:
my_queries = [
    {"id": "Q-09", "text": "Who has to sign form GR-210?", "relevant": ["GO-07"]},
    {"id": "Q-10", "text": "Когда закрывается приём заявок на грант?", "relevant": ["GO-09", "GO-14"]},
]
all_queries = queries + my_queries
print(len(all_queries), "queries in total")

10 queries in total


## Results for all ten or more queries: top 3 from each retriever

In [9]:
results_B = []
for q in all_queries:
    print(f"\n=== {q['id']}: {q['text']}   (relevant: {q['relevant']})")
    row = {"id": q["id"]}
    for name, fn in [("BM25", bm25_top), ("Emb", emb_top)]:
        top = fn(q["text"])
        hit = any(cid in q["relevant"] for cid, _ in top)
        row[name] = hit
        row[name + "_top"] = top
        print(f"  {name}   hit@3 = {hit}")
        for r, (cid, s) in enumerate(top, 1):
            print(f"    {r}. {cid:6} {s:8.3f}  {first_line(cid)}")
    results_B.append(row)

print("\n\nB3 table (copy into the markdown cell below):\n")
print("| Query | BM25 top 3 (ids) | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |")
print("|---|---|---|---|---|")
for r in results_B:
    b = ", ".join(c for c, _ in r["BM25_top"])
    e = ", ".join(c for c, _ in r["Emb_top"])
    print(f"| {r['id']} | {b} | {'✅' if r['BM25'] else '❌'} | {e} | {'✅' if r['Emb'] else '❌'} |")

print("\nhit@3 total:  BM25 =", sum(r["BM25"] for r in results_B), "/", len(results_B),
      "|  Emb =", sum(r["Emb"] for r in results_B), "/", len(results_B))


=== Q-01: What is form GR-305 used for?   (relevant: ['GO-08', 'GO-12', 'GO-23'])
  BM25   hit@3 = True
    1. GO-08    15.825  Form GR-305 is used to appeal a decision of the Grant Office. It is th
    2. GO-12     8.195  A student who disagrees with a decision may appeal within 14 calendar 
    3. GO-06     8.070  Form GR-104 is the application form for the need-based study grant. It
  Emb   hit@3 = True
    1. GO-08     0.696  Form GR-305 is used to appeal a decision of the Grant Office. It is th
    2. GO-12     0.461  A student who disagrees with a decision may appeal within 14 calendar 
    3. GO-23     0.450  Если студент не согласен с решением, он может подать апелляцию по форм

=== Q-02: My family is poor. Is there financial help for students like me?   (relevant: ['GO-01', 'GO-02', 'GO-03'])
  BM25   hit@3 = False
    1. KZ-17     8.726  The lake currently covers about 16,400 km2 (6,300 sq mi). However, lik
    2. GO-20     8.311  The need-based grant is for full-time underg

## B3 — comparison

| Query | BM25 top 3 (ids) | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |
|---|---|:---:|---|:---:|
| Q-01 | GO-08, GO-12, GO-06 | ✅ | GO-08, GO-12, GO-23 | ✅ |
| Q-02 | KZ-17, GO-20, KZ-25 | ❌ | GO-01, GO-10, GO-20 | ✅ |
| Q-03 | GO-14, KZ-27, KZ-20 | ✅ | GO-14, GO-09, GO-11 | ✅ |
| Q-04 | GO-05, GO-17, GO-19 | ❌ | GO-02, GO-05, GO-17 | ✅ |
| Q-05 | KZ-05, KZ-06, KZ-11 | ✅ | KZ-05, KZ-06, KZ-03 | ✅ |
| Q-06 | KZ-37, KZ-38, KZ-03 | ✅ | KZ-37, KZ-38, KZ-10 | ✅ |
| Q-07 | GO-21, GO-16, GO-01 | ❌ | GO-15, GO-18, GO-01 | ✅ |
| Q-08 | GO-17, GO-05, GO-04 | ✅ | GO-17, GO-18, GO-15 | ✅ |
| Q-09 | GO-07, GO-03, GO-08 | ✅ | GO-07, GO-10, GO-13 | ✅ |
| Q-10 | GO-15, GO-14, GO-23 | ✅ | GO-09, GO-19, GO-14 | ✅ |

**hit@3 over 10 queries: BM25 = 7/10, EmbeddingGemma = 10/10.**

**Both retrievers agree:** which query, and why was it easy?

> **Q-06** ("Where did the Soviet Union test its nuclear weapons?"). Both return KZ-37 and KZ-38 as ranks 1 and 2 (BM25 19.09 / 17.17, embeddings 0.596 / 0.510). It is easy because the query words are literally in the chunks ("Soviet Union", "nuclear", "test" in KZ-38, "Test Site" in KZ-37), and nothing else in the corpus is about that topic, so exact matching and meaning point to the same chunks.

**BM25 did better:** which query, and what exact string did it match that the embedding model blurred?

> In my runs BM25 never beat embeddings on hit@3 (7/10 vs 10/10), and my own Q-09, which I expected BM25 to win, was a tie: both put GO-07 first. The closest case is **Q-01** ("What is form GR-305 used for?"). BM25 matched the exact string "305" (from "GR-305") in GO-08 and ranked it first by a wide margin, 15.83 vs 8.20 for the runner-up, almost double. The embedding model also ranked GO-08 first, but with a much smaller gap (0.696 vs 0.461), because to it "GR-305" is one form-like code among GR-104 / GR-210 / GR-305. Exact matching also has a cost here: BM25's 3rd hit is GO-06 (form GR-104), pulled in only by the shared words "form", "is", "used", "for".

**Embeddings did better:** which query, and what could BM25 not match?

> **Q-07** ("Can the grant be sent to my mother's card?"). Embeddings ranked GO-15 first (0.422); BM25 did not have it in the top 3. GO-15 is the only chunk with the answer, and it is in Russian ("перевод на счёт родителей ... невозможен"), so it shares no token with the English query. BM25's top hit was GO-21 (14.65), matching "can", "be" and "sent" from an unrelated sentence about sending questions through the portal. The multilingual embedding model matched the meaning across languages: "mother's card" ↔ "счёт родителей". The same happens in **Q-02**, a paraphrase: "poor" vs "low-income household" in GO-01. BM25 put KZ-17 (the Aral Sea) first; embeddings found GO-01 at rank 1 (0.445).

> **Q-04 (the archived-rule trap):** BM25 ranked GO-05 first (12.78), the archived 2025 rule, because it contains the exact phrase "the minimum GPA for the grant", and missed the current rule GO-02 entirely. Embeddings put GO-02 first (0.665), but GO-05 was a close second (0.621). Only GO-05's title says it is no longer in force, and neither retriever reads that.

---
# Part C — connect retrieval to the model (1.5 pt)

Model: `gpt-5.6-luna` for every call. Top 3 unless stated.

In [13]:
from openai import OpenAI
import tiktoken

client = OpenAI(default_headers={"Accept-Encoding": "gzip, deflate"})
enc = tiktoken.get_encoding("o200k_base")
MODEL = "gpt-5.6-luna"

SYSTEM = """Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3]."""

def build_context(chunk_ids):
    return "\n\n".join(f"[{i}] {by_id[cid]['title']}\n{by_id[cid]['text']}"
                       for i, cid in enumerate(chunk_ids, 1))

def build_messages(question, chunk_ids=None):
    if chunk_ids is None:
        return [{"role": "user", "content": question}]
    user = f"Context:\n{build_context(chunk_ids)}\n\nQuestion: {question}"
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

def ask(messages):
    r = client.chat.completions.create(model=MODEL, messages=messages)
    return r.choices[0].message.content, r.usage

q = qa[0]
top = emb_top(q["question"])
chunk_ids = [cid for cid, _ in top]
print("Question:", q["id"], q["question"])
print("Retrieved (EmbeddingGemma):", top, "\n")

msgs = build_messages(q["question"], chunk_ids)
for m in msgs:
    print(f"===== {m['role'].upper()} =====\n{m['content']}\n")

ctx = build_context(chunk_ids)
t_sys, t_q, t_ctx = len(enc.encode(SYSTEM)), len(enc.encode(q["question"])), len(enc.encode(ctx))
print("System:", t_sys, "| Question:", t_q, "| Context:", t_ctx, "| Total:", t_sys + t_q + t_ctx)
for cid in chunk_ids:
    print(cid, len(enc.encode(by_id[cid]["text"])), "tokens")
print("Chunks:", len(chunk_ids), "| Context characters:", len(ctx))

answer, usage = ask(msgs)
print("\nusage.prompt_tokens =", usage.prompt_tokens)
print("\nANSWER:\n", answer)

Question: C-01 How much does a student in income band 2 receive, and when is it paid?
Retrieved (EmbeddingGemma): [('GO-04', 0.5845215320587158), ('GO-03', 0.48417532444000244), ('GO-15', 0.4506281316280365)] 

===== SYSTEM =====
Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3].

===== USER =====
Context:
[1] How much the grant pays
The grant amount depends only on the income band, not on the GPA. A student in band 1 receives 250,000 tenge for the academic year. A student in band 2 receives 150,000 tenge for the academic year. The amount is paid in two equal instalments, the first in October and the second in February. There is no partial grant: a student either receives the full amount for their band or nothing.

[2] Income bands
The income band is set from the monthly household income per person, declared on form GR-210. Band 1: below 85,000 tenge per perso

## C1 — the complete prompt, printed before it is sent

| Part of the input | Tokens (tiktoken `o200k_base`) |
|---|---:|
| System instructions | 44 |
| Question | 18 |
| Retrieved context | 302 |
| **Total input** | **364** |
| `usage.prompt_tokens` from the response | **378** |

- Number of retrieved chunks: 3 (GO-04, GO-03, GO-15)
- Tokens per chunk: GO-04 = 90, GO-03 = 104, GO-15 = 85 (chunk text only)
- Total context size (tokens / characters): 302 tokens / 1197 characters (279 tokens of chunk text + ~23 tokens for the "[n] title" headers and blank lines)

**Why do the tiktoken total and `usage.prompt_tokens` differ?**

> 378 vs 364, a gap of 14 tokens. tiktoken counted three raw strings; the API counts the whole formatted conversation. (1) My user message also contains the template text "Context:\n" and "\n\nQuestion: ", which is in none of the three parts I counted. (2) The chat format wraps each message (system, user) in special tokens that mark its role and its start and end, and adds tokens that start the assistant's reply. This is the same gap as in HW1: the model is billed for the formatted chat, not for the bare text.

**What exactly does retrieval add to the model's input?**

> Literally, three blocks of text pasted into the user message before the question: "[1] How much the grant pays …", "[2] Income bands …", "[3] Выплата гранта …", 302 tokens and 1197 characters. Nothing else. The model never sees the corpus, the other 71 chunks, the chunk ids (GO-04 is not even in the prompt, only its title), or the similarity scores. Without retrieval, the user message would be just the 18-token question.

## C2 — three systems, six questions

In [15]:
results_C = []
for q in qa:
    row = {"id": q["id"], "reference": q["reference"]}
    row["llm"], _ = ask(build_messages(q["question"]))
    row["bm25_ids"] = [c for c, _ in bm25_top(q["question"])]
    row["bm25"], _ = ask(build_messages(q["question"], row["bm25_ids"]))
    row["emb_ids"] = [c for c, _ in emb_top(q["question"])]
    row["emb"], _ = ask(build_messages(q["question"], row["emb_ids"]))
    results_C.append(row)

for r in results_C:
    print(f"\n######## {r['id']}\nREFERENCE: {r['reference']}")
    print(f"\n[LLM only]\n{r['llm']}")
    print(f"\n[BM25 RAG] {r['bm25_ids']}\n{r['bm25']}")
    print(f"\n[Emb RAG] {r['emb_ids']}\n{r['emb']}")


######## C-01
REFERENCE: 150,000 tenge for the academic year, in two equal instalments: October and February.

[LLM only]
Which country or student-support scheme do you mean? “Income band 2” is used by several programs, and the amount and payment date vary by scheme and academic year.

[BM25 RAG] ['GO-04', 'GO-02', 'GO-01']
A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments**: the first in **October** and the second in **February**. [1]

[Emb RAG] ['GO-04', 'GO-03', 'GO-15']
A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments of 75,000 tenge**: one in **October** and one in **February**. [1]

######## C-02
REFERENCE: 14 calendar days from the date on the decision letter, using form GR-305.

[LLM only]
The appeal deadline and form depend on the **grant scheme, funding body, and country**. They should be stated in your refusal/decision letter—often under “appeal,” “review,”

Mark each answer ✓ (agrees with `reference`) or ✗, and give a few words of the answer.

| Question | LLM only | BM25 RAG | Embedding RAG |
|---|---|---|---|
| C-01 | ✗ asks which country / scheme, no answer | ✓ 150,000 ₸ for the year, two equal instalments, Oct + Feb [1] | ✓ 150,000 ₸, two instalments "of 75,000", Oct + Feb [1] |
| C-02 | ✗ generic: "depends on the scheme", "10–30 days" | ✓ 14 calendar days from the decision letter, Form GR-305 [3] | ✓ 14 calendar days, Form GR-305 [1][2] |
| C-03 | ✗ asks which grant, no answer | ✓ GPA 2.67 on a 4.0 scale [2] | ✓ GPA 2.67 on a 4.0 scale [1] |
| C-04 | ✓ 27 Oct 2017 decree; 2025 target, now 2031 | ✓ Oct 2017; planned for 2025, now 2031 [1][2] | ✓ Oct 2017; now due by 2031 [1][2] |
| C-05 | ✓ 1963, "Alma-Ata Institute of National Economy" (old spelling of Almaty) | ✓ 1963, Almaty Institute of National Economy [1] | ✓ 1963, Almaty Institute of National Economy [1] |
| C-06 | ✗ invented US figures (TEACH ~$4,000/year, etc.) | ✓ "I do not know the amount"; separate scheme not described [3] | ✓ cannot apply, grant is undergraduate-only; separate scheme, ask department [1] |
| **agrees with `reference`** | 2/6 | 6/6 | 6/6 |

LLM only is right exactly on the two Wikipedia questions (C-04, C-05), which it saw in training, and gets none of the four handbook questions, which exist only in this corpus.

### C3 — question 1 of 3: C-03

**1. What BM25 retrieved** (top 3, ids and scores): GO-16, GO-02, GO-06, with scores printed by the cell below. Unlike Q-04 in Part B, the archived GO-05 did not come back here, because this question is worded differently ("needed to qualify for the need-based grant").

**2. What EmbeddingGemma retrieved** (top 3, ids and scores): GO-02, GO-20, GO-05, with scores printed by the cell below.

**3. What was sent to the model** (the context blocks): printed by the cell below. The embedding context contains **both** the current rule GO-02 ("cumulative GPA is at least 2.67") and the archived rule GO-05 ("ARCHIVED … the minimum GPA for the grant was 3.0 … no longer in force").

**4. How retrieval changed the answer:** without context the model only asked which grant I meant. With context both RAG systems answered "2.67 on a 4.0 scale", correct and cited (the GO-02 block in both).

**5. What was missed, and was it a retrieval or a generation problem?** Nothing was missed. The risk was a conflicting chunk: the embedding context also contained the outdated 3.0. Generation handled it correctly, because GO-05's own text says it is archived. If retrieval had returned GO-05 **without** GO-02, as BM25 did for Q-04 in Part B (GO-05, GO-17, GO-19), the model would have seen only the 3.0. That would have been a **retrieval** failure, not a generation one.

In [18]:
qid = "C-03"
q = next(x for x in qa if x["id"] == qid)
r = next(x for x in results_C if x["id"] == qid)
print(qid, q["question"], "| relevant:", q["relevant"])
print("BM25 top 3:", [(c, round(s, 3)) for c, s in bm25_top(q["question"])])
print("Emb  top 3:", [(c, round(s, 3)) for c, s in emb_top(q["question"])])
print("\n--- Context sent (BM25 RAG) ---\n" + build_context(r["bm25_ids"]))
print("\n--- Context sent (Embedding RAG) ---\n" + build_context(r["emb_ids"]))

C-03 What is the minimum GPA needed to qualify for the need-based grant? | relevant: ['GO-02']
BM25 top 3: [('GO-16', 15.799), ('GO-02', 14.059), ('GO-06', 13.852)]
Emb  top 3: [('GO-02', 0.629), ('GO-20', 0.569), ('GO-05', 0.545)]

--- Context sent (BM25 RAG) ---
[1] Combining the grant with other support
The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant is reduced so that the total equals the cap.

[2] Who qualifies
An applicant qualifies for the 2026 grant when all three conditions hold at the time of the decision: their cumulative GPA is at least 2.67 on the 4.0 scale; their household is in income band 1 or income band 2; and both a transcript and a copy of the national id card are on file with the Grant Office. Meeting two conditions out o

### C3 — question 2 of 3: C-01

**1. What BM25 retrieved** (top 3, ids and scores): GO-04, GO-02, GO-01, with scores printed by the cell below.

**2. What EmbeddingGemma retrieved** (top 3, ids and scores): GO-04 (0.585), GO-03 (0.484), GO-15 (0.451).

**3. What was sent to the model** (the context blocks): printed in full by the cell below. Both contexts contain GO-04 ("A student in band 2 receives 150,000 tenge … two equal instalments, the first in October and the second in February"). Only the embedding context also has GO-15, the Russian payment chunk.

**4. How retrieval changed the answer:** without context the model only asked which country or scheme I meant, because it has never seen this handbook. With context both RAG answers became correct, specific and grounded, citing [1].

**5. What was missed, and was it a retrieval or a generation problem?** Nothing from the reference was missed. BM25 did not retrieve GO-15, but GO-04 alone holds the full answer. The surprise is in generation: the Embedding-RAG answer says "two equal instalments **of 75,000 tenge**", but 75,000 appears nowhere in the context. The model computed it (150,000 / 2). It is arithmetically right, but it goes beyond the instruction "answer using only the provided context". The facts were in the context, so this is not a retrieval problem; it is a **generation** behaviour: the model added an inferred number.

In [19]:
qid = "C-01"
q = next(x for x in qa if x["id"] == qid)
r = next(x for x in results_C if x["id"] == qid)
print(qid, q["question"], "| relevant:", q["relevant"])
print("BM25 top 3:", [(c, round(s, 3)) for c, s in bm25_top(q["question"])])
print("Emb  top 3:", [(c, round(s, 3)) for c, s in emb_top(q["question"])])
print("\n--- Context sent (BM25 RAG) ---\n" + build_context(r["bm25_ids"]))
print("\n--- Context sent (Embedding RAG) ---\n" + build_context(r["emb_ids"]))

C-01 How much does a student in income band 2 receive, and when is it paid? | relevant: ['GO-04', 'GO-15']
BM25 top 3: [('GO-04', 20.219), ('GO-02', 18.742), ('GO-01', 17.243)]
Emb  top 3: [('GO-04', 0.585), ('GO-03', 0.484), ('GO-15', 0.451)]

--- Context sent (BM25 RAG) ---
[1] How much the grant pays
The grant amount depends only on the income band, not on the GPA. A student in band 1 receives 250,000 tenge for the academic year. A student in band 2 receives 150,000 tenge for the academic year. The amount is paid in two equal instalments, the first in October and the second in February. There is no partial grant: a student either receives the full amount for their band or nothing.

[2] Who qualifies
An applicant qualifies for the 2026 grant when all three conditions hold at the time of the decision: their cumulative GPA is at least 2.67 on the 4.0 scale; their household is in income band 1 or income band 2; and both a transcript and a copy of the national id card are on file with th

### C3 — question 3 of 3: C-06

**1. What BM25 retrieved** (top 3, ids and scores): GO-16, GO-01, GO-20, with scores printed by the cell below.

**2. What EmbeddingGemma retrieved** (top 3, ids and scores): GO-20, GO-16, GO-01, with scores printed by the cell below.

**3. What was sent to the model** (the context blocks): printed by the cell below. Both contexts contain GO-20 ("Master's and doctoral students are supported by a separate scheme, which this handbook does not describe"), plus GO-01 and GO-16, which contain other rules and amounts (one grant per year; a 400,000 tenge cap).

**4. How retrieval changed the answer:** LLM only gave a long, confident answer about the US (Pell, FSEOG, TEACH up to ~$4,000/year), none of which is in this corpus. BM25 RAG answered "I do not know the amount … separate scheme, which this handbook does not describe" [3]. Embedding RAG answered that a master's student cannot apply for this undergraduate grant and should ask their department [1]. Both are grounded and invent no number, which is the only correct response.

**5. What was missed, and was it a retrieval or a generation problem?** Nothing was missed, because there is no amount to find. Retrieval brought GO-20 back in both systems, which gave the model the evidence to decline. Generation followed the system instruction and did not borrow the 400,000 cap from GO-16, which it could easily have misused. Embedding RAG does not literally say "I don't know", but it correctly answers that this grant does not apply. **No failure**, neither retrieval nor generation.

In [20]:
qid = "C-06"
q = next(x for x in qa if x["id"] == qid)
r = next(x for x in results_C if x["id"] == qid)
print(qid, q["question"], "| relevant:", q["relevant"])
print("BM25 top 3:", [(c, round(s, 3)) for c, s in bm25_top(q["question"])])
print("Emb  top 3:", [(c, round(s, 3)) for c, s in emb_top(q["question"])])
print("\n--- Context sent (BM25 RAG) ---\n" + build_context(r["bm25_ids"]))
print("\n--- Context sent (Embedding RAG) ---\n" + build_context(r["emb_ids"]))

C-06 I am a master's student. How much need-based grant can I get? | relevant: ['GO-20']
BM25 top 3: [('GO-16', 16.996), ('GO-01', 14.173), ('GO-20', 13.72)]
Emb  top 3: [('GO-20', 0.666), ('GO-16', 0.616), ('GO-01', 0.554)]

--- Context sent (BM25 RAG) ---
[1] Combining the grant with other support
The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant is reduced so that the total equals the cap.

[2] About the need-based study grant 2026
The need-based study grant 2026 is run by the Grant Office for full-time undergraduate students. It exists so that a student from a low-income household can pay for study costs without taking on paid work that harms their studies. One grant is awarded per student per academic year. The grant is a gift, not a loan:

---

## Final questions

**1. What is the main difference between BM25 and embedding retrieval?**

> BM25 scores exact token overlap: how often each query word appears in a chunk, weighted by how rare the word is and normalised by chunk length. Embedding retrieval turns the query and every chunk into vectors and ranks by cosine similarity, so it compares meaning rather than spelling. Part A shows it directly: with no exact token match BM25 gave all three documents 0 (A1a), while cosine still ranked D2 first.

**2. When can BM25 perform better than embeddings?**

> When the query contains a rare exact string: a form code, a name, a number. In Q-01 BM25 matched "305" from "GR-305" and put GO-08 first with 15.83 vs 8.20 for the runner-up; in Q-05 the rare names "Astana"/"Nur-Sultan" gave KZ-05 15.73 vs 6.44. It is also cheap, needs no model, and is easy to explain. In my ten queries, though, it never beat embeddings on hit@3.

**3. When can embeddings perform better than BM25?**

> When the words differ but the meaning is the same: paraphrases, synonyms, other languages. Q-02 says "poor" while GO-01 says "low-income"; Q-07 asks in English but the answer GO-15 is in Russian; Q-10 asks in Russian and embeddings found the English GO-09 first. BM25 missed GO-15 in Q-07 and the right chunks in Q-02 entirely, because there was no shared token.

**4. Why is cosine similarity used with embeddings?**

> Cosine measures the angle between two vectors and ignores their length, so a short query and a long chunk are compared by direction, which is where an embedding model puts the meaning, not by magnitude. It is bounded between −1 and 1, which makes scores comparable across queries, and for normalised vectors it is just a dot product, which is cheap. In A2, D1 and D2 had very different lengths (0.81 vs 1.12) but nearly the same direction as the query (0.996 vs 0.998).

**5. What information is actually sent to an LLM in a RAG system?**

> Only text in the prompt: the system instruction, then a user message that contains the retrieved chunks as numbered blocks ("[1] title + text …") followed by the question. In C1 that was 44 + 18 + 302 = 364 tokens by tiktoken, and 378 billed by the API after chat formatting. No chunk ids, no scores, no other documents are sent.

**6. Does the LLM search the original documents itself?**

> No. My code does the search (BM25 or EmbeddingGemma over 74 chunks held in memory), picks the top 3 and pastes their text into the prompt. The model only continues that prompt. That is why LLM-only could not answer C-01, C-02, C-03 or C-06 at all, but answered them correctly once the chunk text was pasted in.

**7. What happens if retrieval returns an irrelevant chunk?**

> It costs tokens and gives the model something wrong to use. If the right chunk is also there, the model can ignore the noise: in C-02, BM25's top hit was the archived GO-05, and in C-03 the embedding context also contained GO-05 (GPA 3.0), yet both answers were correct. If the irrelevant chunk replaces the right one, the answer goes wrong. For Q-04, BM25 returned GO-05, GO-17, GO-19 and no GO-02, so a model given that context would have had only the outdated 3.0.

**8. Why can increasing Top-K sometimes improve RAG and sometimes make it worse?** Report K = 1, 3 and 8 on one question.

> On C-03 with EmbeddingGemma (cell below): **K=1** [GO-02], 180 prompt tokens, "2.67". **K=3** [GO-02, GO-20, GO-05], 346 tokens, "2.67". **K=8** [GO-02, GO-20, GO-05, GO-16, GO-01, GO-06, GO-19, GO-17], 816 tokens, "2.67". The answer never changed, because the right chunk was already at rank 1. Going from K=1 to K=8 only multiplied the input by 4.5 and added noise, including the archived GO-05 with its conflicting 3.0. A larger K helps when the right chunk is ranked lower or the answer is split across chunks (in Q-10 GO-14 was only 3rd; in C-01 the amount is in GO-04 and the payment rule in GO-15). It hurts by adding conflicting or irrelevant text, cost and latency.

In [21]:
q = next(x for x in qa if x["id"] == "C-03")
for k in [1, 3, 8]:
    cids = [c for c, _ in emb_top(q["question"], k=k)]
    ans, u = ask(build_messages(q["question"], cids))
    print(f"\n--- K={k}  chunks={cids}  prompt_tokens={u.prompt_tokens}\n{ans}")


--- K=1  chunks=['GO-02']  prompt_tokens=180
The minimum GPA needed is **2.67 on a 4.0 scale**. [1]

--- K=3  chunks=['GO-02', 'GO-20', 'GO-05']  prompt_tokens=346
The minimum GPA needed is **2.67 on a 4.0 scale**. [1]

--- K=8  chunks=['GO-02', 'GO-20', 'GO-05', 'GO-16', 'GO-01', 'GO-06', 'GO-19', 'GO-17']  prompt_tokens=816
The minimum GPA needed to qualify is **2.67 on the 4.0 scale**. [1]


**9. Is a vector database required to build a RAG system?**

> No. My notebook stored all 74 chunk vectors in a plain NumPy array (`doc_emb` from `encode_document`) and compared the query to every one of them with `model.similarity`; BM25 lived in an in-memory `BM25Okapi` object. A vector database becomes useful at millions of chunks (approximate nearest-neighbour search, persistence, metadata filtering), but it is not part of what makes RAG work.

**10. Which retrieval approach worked better for this corpus, and what evidence supports that?**

> EmbeddingGemma. hit@3 over my ten queries: **EmbeddingGemma 10/10, BM25 7/10**. BM25 missed Q-02 (paraphrase), Q-04 (it ranked the archived GO-05 first and missed GO-02) and Q-07 (the answer is only in Russian). In C2 both RAG systems agreed with the reference on **6/6** questions, against **2/6** for LLM only, so on these six questions the generation step tolerated BM25's noisier contexts. The retrieval advantage is clear in Part B; the end-to-end answers were equal.

---

## Conclusion

Next I would build **hybrid retrieval**: run BM25 and EmbeddingGemma together and merge their rankings (e.g. reciprocal rank fusion). BM25 was strongest on exact codes like "GR-305", and embeddings were the only way to reach paraphrases and the Russian and Kazakh chunks, so each covers the other's misses. I would also add **metadata to the chunks** (in force / archived, language, valid year) and filter out archived rules before ranking, because GO-05 kept appearing in the top 3 for GPA questions only because it uses the student's exact words. Finally I would tighten the system prompt to forbid derived numbers (the "75,000" in C-01) and evaluate on a larger set of questions with known answers, since six questions were too few to separate the two RAG systems.